# Subject Category Leakage Check — ISOT



## 1. Load Data


In [1]:
import pandas as pd

train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("val.csv")
test_df = pd.read_csv("test.csv")

full_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
print("Full dataset shape:", full_df.shape)
print("Columns:", full_df.columns.tolist())
full_df.head()


Full dataset shape: (38635, 8)
Columns: ['title', 'text', 'subject', 'date', 'label', 'title_word_count', 'text_word_count', 'label_id']


,title,text,subject,date,label,title_word_count,text_word_count,label_id
0,MONTEL WILLIAMS Says Oregon Patriots Are “Unde...,"He s pro-gun, pro-military, anti-fracking and ...",Government News,"Jan 5, 2016",fake,17,125,1
1,Trump says he will get intelligence briefing o...,WASHINGTON (Reuters) - President-elect Donald ...,politicsNews,"January 4, 2017",real,12,327,0
2,Philippine lawmakers reject last left-wing cab...,MANILA (Reuters) - Philippine lawmakers on Wed...,worldnews,"September 6, 2017",real,7,263,0
3,NEW VIDEO Of United Airlines Passenger Emerges...,Watch here:@United overbook #flight3411 and de...,left-news,"Apr 12, 2017",fake,26,543,1
4,Anti-immigrant AfD alarms German Jews as elect...,"FRANKFURT/BERLIN (Reuters) - Salomea Genin, an...",worldnews,"September 20, 2017",real,8,779,0


## 2. Subject Distribution by Label


In [2]:
subject_label_counts = full_df.groupby(["subject", "label"]).size().unstack(fill_value=0)
subject_label_pct = subject_label_counts.div(subject_label_counts.sum(axis=1), axis=0)

print("Subject distribution by label (row percentages):")
print(subject_label_pct.round(4))


Subject distribution by label (row percentages):
label            fake  real
subject                    
Government News   1.0   0.0
Middle-east       1.0   0.0
News              1.0   0.0
US_News           1.0   0.0
left-news         1.0   0.0
politics          1.0   0.0
politicsNews      0.0   1.0
worldnews         0.0   1.0


## 3. Subject "Purity" — How Dominated by One Label Is Each Subject?


In [3]:
subject_purity = subject_label_pct.max(axis=1)
print("Subject purity (fraction belonging to the majority label per subject):")
print(subject_purity.sort_values(ascending=False).round(4))

overall_purity = (subject_label_counts.max(axis=1).sum()) / subject_label_counts.sum().sum()
print(f"\nIf you predicted 'majority label per subject' with zero article content, you'd get {overall_purity:.2%} accuracy.")


Subject purity (fraction belonging to the majority label per subject):
subject
Government News    1.0
Middle-east        1.0
News               1.0
US_News            1.0
left-news          1.0
politics           1.0
politicsNews       1.0
worldnews          1.0
dtype: float64

If you predicted 'majority label per subject' with zero article content, you'd get 100.00% accuracy.


## 4. Trivial Classifier: Predict Label from Subject Alone

Trains a Logistic Regression using ONLY the one-hot encoded subject category — no article text at all.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X_subject = full_df[["subject"]]
y_subject = full_df["label_id"]

X_train_subj, X_test_subj, y_train_subj, y_test_subj = train_test_split(
    X_subject, y_subject, test_size=0.15, random_state=42, stratify=y_subject
)

encoder = OneHotEncoder(handle_unknown="ignore")
X_train_enc = encoder.fit_transform(X_train_subj)
X_test_enc = encoder.transform(X_test_subj)

subject_only_clf = LogisticRegression(max_iter=1000, random_state=42)
subject_only_clf.fit(X_train_enc, y_train_subj)

subject_only_preds = subject_only_clf.predict(X_test_enc)
subject_only_acc = accuracy_score(y_test_subj, subject_only_preds)

print(f"Accuracy using ONLY the 'subject' field (no article text at all): {subject_only_acc:.4f}")
print()
print(classification_report(y_test_subj, subject_only_preds, target_names=["real", "fake"]))


Accuracy using ONLY the 'subject' field (no article text at all): 1.0000

              precision    recall  f1-score   support

        real       1.00      1.00      1.00      3179
        fake       1.00      1.00      1.00      2617

    accuracy                           1.00      5796
   macro avg       1.00      1.00      1.00      5796
weighted avg       1.00      1.00      1.00      5796



## 5. Interpretation

This confirms "subject" is a second major leakage source in ISOT, independent of the Reuters dateline — the model doesn't need to read any article content to guess the label correctly most of the time. This would help explain why the cleaned models (Reuters tag stripped, retrained from scratch) still scored much higher - removing the Reuters tag alone wasn't enough, since subject category remained as an easy fallback shortcut.
